# LA 05a — Full signal context, −240 to +240 minutes

**Descriptive historical event study, NOT an alpha signal.** Each event is aligned to `t=0 = completed second-touch signal candle CLOSE`. The first touch, actual entry print, and eventual exit are separate markers. **Pre-signal bars cannot know the future event's outcome or even necessarily its eventual liquidity level.**

The market path is displayed for the entire +240m after the signal **even when the simulated position has exited**. Trade-active counts are separate; lines after exit are not trade P&L. Outcomes come from the frozen A04 primary A03d fixed-stop 240m cohort. BTC and ETH are shown separately; `target/sigma60 <=1` versus `>1` context is available for subgroup comparisons. No raw trade ticks required.

In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Run Jupyter inside repo'
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a05a import VERSION, Config, run
assert VERSION=='LA-A05a-0.1.0', 'git pull and restart kernel'
TEST=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests',
    '-p','test_liquidity_absorption_stage_a05a.py','-q'],cwd=str(ROOT),capture_output=True,text=True)
print(TEST.stdout[-3000:],TEST.stderr[-5000:])
assert TEST.returncode==0, 'A05a unit tests failed'
DATA=binance_root(ROOT)
OUT=ROOT/'notebooks'/'Liquidity_Absorption'/'05a'
print('A05a:',VERSION,'| Binance 1m:',DATA,'| Output:',OUT)

In [ ]:
decision=run(ROOT,DATA,OUT,Config())
print('Completed descriptive signal context study:')
display(decision)
assert decision['signals']==452 and decision['minute_rows']==452*481
assert decision['missing_bar_rows']>=0

## Data integrity and original outcome groups

Keep all 452 candidate IDs and separate target, stop, timeout, targetless, and censored. `05a_coverage.csv` shows full 240m pre/post candle availability. Missing minutes are not imputed; rolling features require consecutive bars.

In [ ]:
qc=pd.read_csv(OUT/'05a_coverage.csv')
events=pd.read_csv(OUT/'05a_cohort_metadata.csv')
curves=pd.read_csv(OUT/'05a_median_context_curves.csv')
landmarks=pd.read_csv(OUT/'05a_context_checkpoints.csv')
phases=pd.read_csv(OUT/'05a_phase_panel.csv')
phase_summary=pd.read_csv(OUT/'05a_phase_summary.csv')
print('Outcome counts by symbol:')
display(events.groupby(['symbol','eventual_outcome']).size().rename('events').reset_index())
print('Market data completeness per symbol:')
display(qc.groupby('symbol')[['complete_pre_240','complete_post_240']].agg(['sum','count']))
print('Fixed clock checkpoints:')
display(landmarks.query('feasibility_band=="ALL" and relative_min in [-240,-60,-15,0,15,60,240]')[[
    'symbol','eventual_outcome','relative_min','n_candidates','n_price_present','n_trade_active',
    'median_signed_close_from_signal_bp','median_flow_alignment_15m','median_rv_15m_bp']].round(3))

## Formation, approach, and post-signal phases

These **fixed phase windows** summarize what happened, not which strategy to trade. Before the signal, price relative to the *eventually identified* liquidity level is a retrospective visualization. Do not use the later label, target, stop, or level before it was actually known.

In [ ]:
display(phase_summary.query('feasibility_band=="ALL"')[[
  'symbol','eventual_outcome','phase','n_events','n_complete',
  'median_signed_price_change_bp','median_aligned_flow_fraction',
  'median_median_quote_vs_prior60','median_median_rv_15m_bp',
  'median_zone_overlap_fraction']].round(3))

## Visual inspection of the entire market event window

All charts use **−240 ... +240 minutes** from signal close. An exited trade remains in market-path charts, so do not interpret the right-hand side as live P&L or a surviving-trade trajectory. Heatmaps and deterministic earliest-complete event examples help expose heterogeneity hidden by medians.

In [ ]:
for pic in (
    '05a_price_paths.png', '05a_level_context.png', '05a_rv_context.png',
    '05a_activity_context.png', '05a_flow_context.png',
    '05a_BTCUSDT_event_heatmap.png', '05a_ETHUSDT_event_heatmap.png',
    '05a_example_event_timelines.png'):
    print(pic)
    display(Image(filename=str(OUT/pic)))

## Decision gate

1. Do not mistake a retrospectively chosen liquidity level for a level already identifiable at −240m; first-touch and level-source timestamps matter.
2. Do not use post-exit price changes for trade P&L or exposure.
3. Correlation of pre-signal pattern with later TARGET versus STOP does not identify passive absorption or imply profitability.
4. Geometry matters: keep the A03d target-to-VD18-sigma context as a stratification, **not** as a pre-formation feature.
5. If interesting pre-signal structures recur, freeze candidate features at t=0 and evaluate later on untouched data, grouped by asset, level family, and day.

**Next:** commit the executed notebook and `05a/` results to GitHub main for review. No earlier simulator, model, or data collector is modified.